# 03 — SARSA (On-Policy TD Control)
**Week 5 | Model-Free Learning**

SARSA learns Q(s,a) — action values — using the **actual next action taken**:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[ R + \gamma \; Q(s', \underbrace{a'}_{\text{action actually taken}}) - Q(s,a) \Big]$$

The name comes from the quintuple: **(S, A, R, S', A')**  
On-policy: the policy being improved is the same one being followed.

In [ ]:
try:
    import gymnasium as gym
except ImportError:
    import subprocess, sys; subprocess.check_call([sys.executable,'-m','pip','install','gymnasium','-q']); import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

env = gym.make('Taxi-v3')
n_s = env.observation_space.n; n_a = env.action_space.n
print(f"Taxi-v3: {n_s} states, {n_a} actions")

## 1. SARSA Implementation

In [ ]:
def sarsa(env, n_episodes=10_000, alpha=0.1, gamma=0.99,
          eps_start=1.0, eps_end=0.01, eps_decay=0.999):
    Q = np.zeros((n_s, n_a))
    eps = eps_start
    returns_per_ep = []
    steps_per_ep = []

    def eps_greedy(s):
        if np.random.rand() < eps: return env.action_space.sample()
        return np.argmax(Q[s])

    for ep in range(n_episodes):
        s, _ = env.reset()
        a = eps_greedy(s)
        ep_return = 0.0
        steps = 0
        done = False

        while not done:
            ns, r, term, trunc, _ = env.step(a)
            done = term or trunc
            na = eps_greedy(ns)

            # SARSA update — uses next action a' that was CHOSEN (on-policy)
            Q[s,a] += alpha * (r + gamma * Q[ns,na] * (not done) - Q[s,a])

            s, a = ns, na
            ep_return += r
            steps += 1

        eps = max(eps_end, eps * eps_decay)
        returns_per_ep.append(ep_return)
        steps_per_ep.append(steps)

    return Q, returns_per_ep, steps_per_ep

print("Training SARSA on Taxi-v3...")
Q_sarsa, returns_sarsa, steps_sarsa = sarsa(env, n_episodes=15_000)
print(f"Done. Final ε ≈ 0.01")

In [ ]:
window = 300
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, data, label, color in [
    (axes[0], returns_sarsa, 'Avg Return', 'steelblue'),
    (axes[1], steps_sarsa,   'Steps/Episode', 'darkorange')]:
    rolling = np.convolve(data, np.ones(window)/window, mode='valid')
    ax.plot(rolling, color=color, linewidth=1.5)
    ax.set_xlabel('Episode'); ax.set_ylabel(label)
    ax.set_title(f'SARSA — {label} (rolling {window})')
plt.tight_layout(); plt.show()

## 2. Visualise the Q-Table

In [ ]:
action_names = ['South','North','East','West','Pickup','Dropoff']

# Show Q-values for first 20 states
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(Q_sarsa[:30, :], cmap='RdYlGn', aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(n_a)); ax.set_xticklabels(action_names, rotation=20, ha='right')
ax.set_ylabel('State (first 30)'); ax.set_title('Q-Table (SARSA) — first 30 states')
plt.tight_layout(); plt.show()

## 3. Evaluate the Greedy Policy

In [ ]:
def evaluate(env, Q, n_episodes=500):
    rewards = []
    for _ in range(n_episodes):
        s, _ = env.reset(); ep_r = 0; done = False
        for _ in range(200):
            a = np.argmax(Q[s])
            s, r, term, trunc, _ = env.step(a)
            ep_r += r; done = term or trunc
            if done: break
        rewards.append(ep_r)
    return np.mean(rewards), np.std(rewards)

mean_r, std_r = evaluate(env, Q_sarsa)
print(f"SARSA greedy policy: mean return = {mean_r:.2f} ± {std_r:.2f}")
print(f"(Random policy baseline is around -200)")

## ✅ Exercises
1. Change α from 0.1 to 0.5. Does SARSA learn faster or does it become unstable?
2. Use a **decaying ε** schedule (start=1.0, end=0.01, linear decay). Compare to exponential decay.
3. **Challenge**: implement **SARSA(λ)** with eligibility traces. Does it learn faster than SARSA(0)?